# Lesson 01 - Pengenalan kepada Ejen AI

Selamat datang ke pelajaran pertama dalam kursus **Ejen AI untuk Pemula**!

Sebuah **ejen AI** adalah program yang menggunakan model bahasa besar (LLM) sebagai enjin penaakulan dan boleh mengambil *tindakan* di dunia nyata — memanggil API, membuat pertanyaan pangkalan data, atau menjalankan kod — untuk mencapai matlamat bagi pihak pengguna.

Dalam buku nota ini anda akan membina ejen pertama anda: sebuah **Ejen Pelancongan** yang mengesyorkan destinasi bercuti. Sepanjang proses, anda akan belajar bagaimana untuk:

1. Bersambung ke Perkhidmatan Ejen Azure AI Foundry menggunakan **Rangka Kerja Ejen Microsoft**.
2. Memberi ejen sebuah **alat** — sebuah fungsi Python biasa yang boleh ia panggil.
3. Menjalankan ejen dan memeriksa tindak balasnya.
4. Menstrim tindak balas ejen satu token demi satu token.


## Setup

Sebelum menjalankan notebook ini, pastikan anda telah:

1. **Projek Azure AI Foundry** dengan model sembang yang telah disebarkan (contohnya `gpt-4o-mini`).
2. **Log masuk dengan Azure CLI** — jalankan `az login` di terminal anda.
3. **Tetapkan pemboleh ubah persekitaran yang diperlukan:**
   - `AZURE_AI_PROJECT_ENDPOINT` — titik akhir projek Azure AI Foundry anda.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nama model anda yang telah disebarkan.

Sel selanjutnya memasang pakej Python yang anda perlukan.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mencipta Ejen Pertama Anda

Satu ejen memerlukan dua perkara:

- **Arahan** yang memberitahunya *siapa dia* dan *bagaimana untuk berkelakuan* (promp sistem).
- **Alat** — fungsi Python dihias dengan `@tool` yang boleh dipanggil oleh ejen untuk mendapatkan maklumat atau melakukan tindakan.

Di bawah kami mentakrifkan satu alat ringkas yang mengembalikan senarai destinasi percutian popular. Ejen akan menggunakan alat ini apabila pengguna meminta cadangan perjalanan.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

Untuk pengalaman yang lebih interaktif, anda boleh **strim** tindak balas ejen. Daripada menunggu balasan penuh, ejen akan menyerahkan potongan teks ketika ia dijana. Ini amat berguna dalam antara muka perbualan di mana anda mahu memaparkan keluaran secara masa nyata.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Summary

Dalam pelajaran ini anda belajar bagaimana untuk:

- **Mewujudkan penyedia** yang bersambung ke Azure AI Foundry Agent Service melalui `AzureAIProjectAgentProvider`.
- **Mentakrifkan alat** menggunakan dekorator `@tool` supaya ejen boleh memanggil fungsi Python anda.
- **Menjalankan ejen** dengan mesej pengguna dan mencetak responnya.
- **Menstrim jawapan** untuk output masa nyata.

Dalam pelajaran seterusnya kita akan meneroka kerangka kerja agentik dengan lebih mendalam dan belajar bagaimana untuk memberikan ejen alat yang lebih berkuasa serta keupayaan penaakulan pelbagai langkah.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk memastikan ketepatan, sila ambil perhatian bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidakakuratan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat yang kritikal, disyorkan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggungjawab atas sebarang salah faham atau tafsiran yang salah yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
